# **Ejemplo 5:**

Caso de alta dimensionalidad, se ejemplifica con una ecuacuación de Black-Scholes

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.autograd import grad

## **1. Arquitectura de las Redes Neuronales (SPINN, Separable PINNs)**

En este caso, al tratarse de un problema de alta dimensionalidad ($d = 10$), la PINN debe ser separable (aproximar la función de solución de alta dimensión como una suma de productos de funciones de menor dimensión).

In [ ]:
class SPINN(nn.Module):
    def __init__(self, d, R=50, hidden_dim=64):
        super().__init__()
        self.d = d
        self.R = R

        # Red para el tiempo t
        self.net_t = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, R)
        )
        # Redes para los subyacentes s_i
        self.net_s = nn.ModuleList([
            nn.Sequential(nn.Linear(1, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, R))
            for _ in range(d)
        ])

        # Capa final para combinar
        self.final_layer = nn.Linear(R, 1, bias=False)

    def forward(self, s, t):
        # s shape: (batch_size, d), t shape: (batch_size, 1)
        out_t = self.net_t(t)

        # Multiplicación tensorial (Hadamard) de las salidas separables
        out_s = torch.ones_like(out_t)
        for i in range(self.d):
            out_s = out_s * self.net_s[i](s[:, i:i+1])

        res = out_t * out_s
        return self.final_layer(res)

## **2. Configuración y planteamiento del SPINN (Separable PINN):**

La ecuación de Black-Scholes a tratar está dada por:
$$\frac{\partial u}{\partial t} + \frac{1}{2} \sum_{i,j=1}^d \rho_{ij} \sigma_i \sigma_j s_i s_j \frac{\partial^2 u}{\partial s_i \partial s_j} + r \sum_{i=1}^d s_i \frac{\partial u}{\partial s_i} - ru = 0$$

donde el valor de la opción se define por la función de pago para una opción de cesta europea:

$$u(\mathbf{s}, T) = \max \left( \frac{1}{d} \sum_{i=1}^d s_i - K, 0 \right)$$

In [ ]:
d = 10         #Dimensión
r = 0.05       #Tasa libre de riesgo
K = 100.0      #Precio de ejercicio (Strike)
T = 1.0
S0 = 100.0     #Precio inicial de los subyacentes

#Volatilidades (constantes para simplificar) y Matriz de correlación
sigma = torch.full((d,), 0.2)
rho = torch.full((d, d), 0.5)
rho.fill_diagonal_(1.0)

#Método de Montecarlo para realizar la comparación (Implementación con IA):
def monte_carlo_basket(d, r, sigma, rho, K, T, S0, num_paths=100000):
  np.random.seed(42)

  # Cholesky para variables correlacionadas
  L = np.linalg.cholesky(rho.numpy())
  Z = np.random.standard_normal((num_paths, d))
  W = Z @ L.T  # Movimiento Browniano correlacionado

  # Simulación de Black-Scholes (GBM) exacto a tiempo T
  drift = (r - 0.5 * sigma.numpy()**2) * T
  diffusion = sigma.numpy() * np.sqrt(T) * W
  S_T = S0 * np.exp(drift + diffusion)

  # Payoff de la Basket Option Europea
  basket_mean = np.mean(S_T, axis=1)
  payoff = np.maximum(basket_mean - K, 0)

  # Valor presente
  price = np.exp(-r * T) * np.mean(payoff)
  return price

#Función de pérdida:
def pde_loss(model, s, t, r, sigma, rho):
  u = model(s, t)

  # Primeras derivadas
  du_dt = grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
  du_ds = grad(u, s, grad_outputs=torch.ones_like(u), create_graph=True)[0]

  #Segundas derivadas (Diagonal del Hessiano y Cruzadas)
  #Nota: Para d=10 de forma robusta, calculamos iterativamente
  batch_size = s.shape[0]
  term_diff = torch.zeros(batch_size, 1, device=s.device)

  for i in range(d):
      for j in range(d):
        d2u_dsidsj = grad(du_ds[:, i], s, grad_outputs=torch.ones_like(du_ds[:, i]), create_graph=True)[0][:, j:j+1]
        cov = rho[i, j] * sigma[i] * sigma[j] * s[:, i:i+1] * s[:, j:j+1]
        term_diff += 0.5 * cov * d2u_dsidsj

  term_drift = r * torch.sum(s * du_ds, dim=1, keepdim=True)
  term_discount = -r * u

  residual = du_dt + term_diff + term_drift + term_discount
  return torch.mean(residual**2)

def terminal_loss(model, s, K, T_val):
  t = torch.full((s.shape[0], 1), T_val, device=s.device, requires_grad=True)
  u_pred = model(s, t)
  s_mean = torch.mean(s, dim=1, keepdim=True)
  payoff = torch.clamp(s_mean - K, min=0.0)
  return torch.mean((u_pred - payoff)**2)

#Entrenamiento:
#Obtener precio exacto (Monte Carlo)
print("Calculando precio con Monte Carlo...")
mc_price = monte_carlo_basket(d, r, sigma, rho, K, T, S0)
print(f"Precio Monte Carlo (Ground Truth): {mc_price:.4f}\n")

#Configurar PINN
model = SPINN(d=d, R=50)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

batch_size = 1000
epochs = 1000 #Reducido

print("Iniciando entrenamiento del PINN (SPINN)")

for epoch in range(epochs):
  optimizer.zero_grad()

  #Puntos de colocación interiores (s aleatorios cerca de S0, t aleatorio)
  s_col = S0 * torch.exp(torch.randn(batch_size, d) * 0.2)
  s_col.requires_grad = True
  t_col = torch.rand(batch_size, 1) * T
  t_col.requires_grad = True

  #Puntos frontera (t = T)
  s_term = S0 * torch.exp(torch.randn(batch_size, d) * 0.2)

  #Calcular pérdidas
  loss_eq = pde_loss(model, s_col, t_col, r, sigma, rho)
  loss_ic = terminal_loss(model, s_term, K, T)
  loss = loss_eq + 10.0 * loss_ic #Ponderación mayor a la condición final

  loss.backward()
  optimizer.step()

  if (epoch + 1) % 250 == 0:
    print(f"Epoch {epoch+1}/{epochs} | Loss PDE: {loss_eq.item():.4f} | Loss Terminal: {loss_ic.item():.4f}")

#Comparación Final
s_test = torch.full((1, d), S0)
t_test = torch.zeros((1, 1))
pinn_price = model(s_test, t_test).item()

print("\n" + "="*40)
print("RESULTADOS FINALES")
print("="*40)
print(f"Precio Monte Carlo (d=10):  {mc_price:.4f}")
print(f"Precio estimado por PINN:   {pinn_price:.4f}")
print(f"Diferencia absoluta:        {abs(mc_price - pinn_price):.4f}")
print("========================================")

Calculando precio con Monte Carlo...
Precio Monte Carlo (Ground Truth): 8.5230

Iniciando entrenamiento del PINN (muestra de epochs)...
Epoch 250/1000 | Loss PDE: 1.0356 | Loss Terminal: 3.1657
Epoch 500/1000 | Loss PDE: 0.7176 | Loss Terminal: 2.6724
Epoch 750/1000 | Loss PDE: 0.5465 | Loss Terminal: 2.6198
Epoch 1000/1000 | Loss PDE: 0.4643 | Loss Terminal: 2.2119

RESULTADOS FINALES
Precio Monte Carlo (d=10):  8.5230
Precio estimado por PINN:   8.2673
Diferencia absoluta:        0.2557
